In [ ]:
!apt-get update -y
!apt-get install -y ffmpeg
!pip install -q librosa soundfile tqdm requests torch

import os
import zipfile
import requests
import shutil
from pathlib import Path
from tqdm import tqdm
import numpy as np
import torch
import librosa
from google.colab import files

YANDEX_URL = "YANDEX DISK LINK"
ZIP_PATH = "input.zip"

INPUT_DIR = "data"
OUTPUT_DIR = "stage1_filtered"


def download_yandex_disk(url, output_path):
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"
    r = requests.get(api_url, params={"public_key": url})
    download_url = r.json()["href"]

    with requests.get(download_url, stream=True) as r:
        with open(output_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

def unzip(zip_path, extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)

vad_model, vad_utils = torch.hub.load(
    'snakers4/silero-vad',
    'silero_vad',
    trust_repo=True
)

(get_speech_timestamps, _, _, _, _) = vad_utils

MIN_DURATION = 1.5
MIN_SPEECH_RATIO = 0.3
MIN_TOTAL_SPEECH = 1.0
RMS_MIN = 0.003
SNR_MIN = 1.3

reasons = {
    "too_short": 0,
    "no_speech": 0,
    "low_speech_ratio": 0,
    "too_noisy": 0,
    "error": 0
}


def get_speech_ratio_silero(y):
    timestamps = get_speech_timestamps(
        torch.from_numpy(y),
        vad_model,
        sampling_rate=16000
    )

    if not timestamps:
        return 0.0, 0.0

    total = 0.0
    for t in timestamps:
        total += (t['end'] - t['start']) / 16000

    return total, total / (len(y) / 16000)


def is_too_noisy(y):
    rms = np.sqrt(np.mean(y**2))
    peak = np.max(np.abs(y)) + 1e-6
    snr_like = peak / (rms + 1e-6)

    if rms < RMS_MIN:
        return True
    if snr_like < SNR_MIN:
        return True

    return False


def check_stage1(path):
    try:
        y, sr = librosa.load(path, sr=16000)
        duration = len(y) / sr

        if duration < MIN_DURATION:
            return False, "too_short"

        total_speech, speech_ratio = get_speech_ratio_silero(y)

        if total_speech < MIN_TOTAL_SPEECH:
            return False, "no_speech"

        if speech_ratio < MIN_SPEECH_RATIO:
            return False, "low_speech_ratio"

        if is_too_noisy(y):
            return False, "too_noisy"

        return True, None

    except:
        return False, "error"


download_yandex_disk(YANDEX_URL, ZIP_PATH)
unzip(ZIP_PATH, INPUT_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)

files_list = []
for root, _, fs in os.walk(INPUT_DIR):
    for f in fs:
        if f.lower().endswith(".wav"):
            files_list.append(os.path.join(root, f))

kept = 0
removed = 0

for path in tqdm(files_list):
    ok, reason = check_stage1(path)

    if ok:
        shutil.copy(path, os.path.join(OUTPUT_DIR, os.path.basename(path)))
        kept += 1
    else:
        removed += 1
        reasons[reason] += 1

print("\nSTAGE 1 STATS")
print(f"Total files: {len(files_list)}")
print(f"Kept: {kept}")
print(f"Removed: {removed}\n")

for k, v in reasons.items():
    print(f"{k}: {v} ({v/len(files_list)*100:.2f}%)")

# ZIP
shutil.make_archive("3_non-urmi_unlabeled", 'zip', OUTPUT_DIR)
files.download("3_non-urmi_unlabeled.zip")